# Step 5 — Detect mangroves, estimate biomass and carbon

Classifies mangrove pixels by thresholds on the indices, applies the allometric model
(Biomass = 250.5 × NDVI − 75.2, Mg/ha, Myanmar Wunbaik Forest, R² = 0.72) and IPCC carbon
accounting (carbon fraction 0.47, CO₂/C = 3.67). Thresholds and coefficients are inputs,
with the values of the single-notebook version as defaults.

| | |
|---|---|
| Six-phase position | Scientific computation |
| W1 Algae Bloom counterpart | `calculate-band` (chlorophyll-a / turbidity formulas) |
| Outputs | `mangrove_mask_file`, `biomass_file` (GeoTIFF), `biomass_summary_file`, `carbon_summary_file` (CSV), `analysis_summary_file` (JSON) |

As in the single-notebook version, a pixel counts for 10 m × 10 m (100 m²).

In [ ]:
import json
from datetime import datetime

import numpy as np
import pandas as pd
import rasterio

In [ ]:
# CWL type annotations (removed by ipython2cwl in the generated tool)
from typing import List, Optional

from ipython2cwl.iotypes import (
    CWLDirectoryPathOutput,
    CWLFilePathInput,
    CWLFilePathOutput,
    CWLFloatInput,
    CWLIntInput,
    CWLMetadata,
    CWLNamespaces,
    CWLRequirement,
    CWLStringInput,
)

In [ ]:
cwl_requirements: CWLRequirement = {
    "ResourceRequirement": {"coresMin": 1, "ramMin": 1024},
}

In [ ]:
cwl_metadata: CWLMetadata = {
    "s:softwareVersion": "0.1.0",
    "s:keywords": ["ospd", "mangrove", "biomass", "carbon"],
    "s:author": [{"class": "s:Person", "s:name": "Cameron Sajedi"}],
    "s:contributor": [
        {"class": "s:Person", "s:name": "Gérald Fenoy", "s:affiliation": "GeoLabs"}
    ],
    "s:codeRepository": "https://github.com/starling-foundries/KindGrove",
    "s:license": "https://spdx.org/licenses/CC-BY-NC-SA-4.0",
    "s:description": "Mangrove detection, allometric biomass and IPCC carbon stock",
}

In [ ]:
cwl_namespaces: CWLNamespaces = {
    "s": "https://schema.org/",
}

## Inputs

In [ ]:
ndvi_file: CWLFilePathInput = "ndvi.tif"
ndwi_file: CWLFilePathInput = "ndwi.tif"
savi_file: CWLFilePathInput = "savi.tif"
stac_item: CWLFilePathInput = "scene_item.json"
ndvi_min: Optional[CWLFloatInput] = 0.3
ndvi_max: Optional[CWLFloatInput] = 0.9
ndwi_min: Optional[CWLFloatInput] = -0.3
savi_min: Optional[CWLFloatInput] = 0.2
biomass_slope: Optional[CWLFloatInput] = 250.5
biomass_intercept: Optional[CWLFloatInput] = -75.2
carbon_fraction: Optional[CWLFloatInput] = 0.47

In [ ]:
def read(path):
    with rasterio.open(path) as src:
        return src.read(1).astype("float64"), src.profile.copy()


ndvi, profile = read(ndvi_file)
ndwi, _ = read(ndwi_file)
savi, _ = read(savi_file)
with open(stac_item) as f:
    scene = json.load(f)
scene_date = datetime.fromisoformat(scene["properties"]["datetime"].replace("Z", "+00:00"))
scene_cloud_cover = scene["properties"].get("eo:cloud_cover")

## Mangrove detection

In [ ]:
mangrove_mask = (
    (ndvi > ndvi_min)  # Vegetated
    & (ndvi < ndvi_max)  # Not upland forest
    & (ndwi > ndwi_min)  # Near water
    & (savi > savi_min)  # Adjusted vegetation
).astype(float)

pixel_area_m2 = 10 * 10
mangrove_pixels = np.sum(mangrove_mask)
mangrove_area_ha = (mangrove_pixels * pixel_area_m2) / 10000
print(f"Detected mangrove area: {mangrove_area_ha:.1f} hectares")
print(f"Coverage: {(mangrove_pixels / mangrove_mask.size * 100):.1f}% of study area")

## Biomass

In [ ]:
biomass = biomass_slope * ndvi + biomass_intercept
biomass_masked = np.where(mangrove_mask > 0, biomass, np.nan)
biomass_masked = np.maximum(biomass_masked, 0)
valid_biomass = biomass_masked[~np.isnan(biomass_masked)]

if len(valid_biomass) > 0:
    mean_biomass = np.mean(valid_biomass)
    median_biomass = np.median(valid_biomass)
    max_biomass = np.max(valid_biomass)
    std_biomass = np.std(valid_biomass)
else:
    print("Warning: No valid biomass estimates")
    mean_biomass = median_biomass = max_biomass = std_biomass = 0
print(f"Mean biomass: {mean_biomass:.1f} Mg/ha")

## Carbon

In [ ]:
pixel_area_ha = (10 * 10) / 10000
total_biomass_mg = np.sum(valid_biomass) * pixel_area_ha if len(valid_biomass) > 0 else 0
carbon_stock_mg = total_biomass_mg * carbon_fraction
co2_equivalent_mg = carbon_stock_mg * 3.67  # CO2 to C ratio
print(f"Total biomass: {total_biomass_mg:,.0f} Mg")
print(f"Carbon stock: {carbon_stock_mg:,.0f} Mg C")

## Outputs

In [ ]:
mask_profile = dict(profile, dtype="uint8", nodata=255, compress="deflate")
mangrove_mask_file: CWLFilePathOutput = "mangrove_mask.tif"
with rasterio.open(mangrove_mask_file, "w", **mask_profile) as dst:
    dst.write(mangrove_mask.astype("uint8"), 1)
    dst.set_band_description(1, "Mangrove mask (1 = mangrove)")

biomass_profile = dict(profile, dtype="float32", nodata=np.nan, compress="deflate")
biomass_file: CWLFilePathOutput = "biomass.tif"
with rasterio.open(biomass_file, "w", **biomass_profile) as dst:
    dst.write(biomass_masked.astype("float32"), 1)
    dst.set_band_description(1, "Above-ground biomass (Mg/ha)")

In [ ]:
biomass_summary_file: CWLFilePathOutput = "biomass_summary.csv"
pd.DataFrame(
    [
        {"Metric": "Mangrove Area (ha)", "Value": f"{mangrove_area_ha:.1f}"},
        {"Metric": "Mean Biomass (Mg/ha)", "Value": f"{mean_biomass:.1f}"},
        {"Metric": "Median Biomass (Mg/ha)", "Value": f"{median_biomass:.1f}"},
        {"Metric": "Max Biomass (Mg/ha)", "Value": f"{max_biomass:.1f}"},
        {"Metric": "Std Deviation (Mg/ha)", "Value": f"{std_biomass:.1f}"},
    ]
).to_csv(biomass_summary_file, index=False)

carbon_summary_file: CWLFilePathOutput = "carbon_summary.csv"
carbon_density = carbon_stock_mg / mangrove_area_ha if mangrove_area_ha > 0 else 0
pd.DataFrame(
    [
        {"Metric": "Total Biomass (Mg)", "Value": f"{total_biomass_mg:,.0f}"},
        {"Metric": "Carbon Stock (Mg C)", "Value": f"{carbon_stock_mg:,.0f}"},
        {"Metric": "CO2 Equivalent (Mg CO2)", "Value": f"{co2_equivalent_mg:,.0f}"},
        {"Metric": "Carbon Density (Mg C/ha)", "Value": f"{carbon_density:.1f}"},
        {"Metric": "Analysis Date", "Value": datetime.now().strftime("%Y-%m-%d")},
        {"Metric": "Scene Date", "Value": scene_date.strftime("%Y-%m-%d")},
        {
            "Metric": "Cloud Cover (%)",
            "Value": f"{scene_cloud_cover:.1f}" if scene_cloud_cover is not None else "N/A",
        },
        {"Metric": "Uncertainty", "Value": "±30%"},
    ]
).to_csv(carbon_summary_file, index=False)

In [ ]:
analysis_summary_file: CWLFilePathOutput = "analysis_summary.json"
with open(analysis_summary_file, "w") as f:
    json.dump(
        {
            "scene_id": scene["id"],
            "scene_datetime": scene["properties"]["datetime"],
            "cloud_cover": scene_cloud_cover,
            "mangrove_area_ha": float(mangrove_area_ha),
            "mean_biomass_mg_ha": float(mean_biomass),
            "biomass_tons": float(total_biomass_mg),
            "carbon_tons": float(carbon_stock_mg),
            "co2_equivalent_tons": float(co2_equivalent_mg),
            "parameters": {
                "ndvi_min": ndvi_min,
                "ndvi_max": ndvi_max,
                "ndwi_min": ndwi_min,
                "savi_min": savi_min,
                "biomass_slope": biomass_slope,
                "biomass_intercept": biomass_intercept,
                "carbon_fraction": carbon_fraction,
            },
        },
        f,
        indent=2,
    )
print("Saved: mangrove_mask.tif, biomass.tif, biomass_summary.csv, carbon_summary.csv, analysis_summary.json")